In [ ]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


In [ ]:
# =============================================================================
# BLOCK 2: MASTER FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Master Feature Engineering ---")

def create_master_features(df_train, df_test):
    # Combine for consistent processing
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

    # --- Part 1: Date Features and Basic Ratios ---
    print("  Creating date and basic ratio features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['age'] = all_data['sale_year'] - all_data['year_built']
    all_data['age'] = all_data['age'].apply(lambda x: max(x, 0))

    # --- Part 2: Brute-Force Interactions (from v1) ---
    print("  Creating brute-force interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # --- Part 3: Advanced Features ---
    print("  Creating advanced ratio and interaction features...")
    epsilon = 1e-6
    all_data['imp_val_per_sqft'] = all_data['imp_val'] / (all_data['sqft'] + epsilon)
    all_data['lot_to_house_ratio'] = all_data['sqft_lot'] / (all_data['sqft'] + epsilon)
    all_data['imp_to_land_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['grade_x_sqft'] = all_data['grade'] * all_data['sqft']
    all_data['grade_x_age'] = all_data['grade'] * all_data['age']

    # --- Part 4: Text Features ---
    print("  Creating TF-IDF features...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf.fit_transform(all_data[col]))
        all_data = pd.concat([all_data, pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])], axis=1)

    # ========================= THE FIX IS HERE =========================
    # --- Part 5: Final Cleanup ---
    print("  Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket', 'year', 'sale_year', 'sale_month']
    
    # Only drop columns that actually exist in the dataframe
    existing_cols_to_drop = [col for col in cols_to_drop if col in all_data.columns]
    all_data = all_data.drop(columns=existing_cols_to_drop)
    # ====================================================================
    
    all_data.fillna(0, inplace=True)

    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train', 'sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train', 'sale_price'])
    
    # Ensure test set has same columns as train set
    X_test = X_test[X.columns]
    
    return X, X_test, test_ids

# --- Run Feature Engineering ---
X, X_test, test_ids = create_master_features(df_train, df_test)
print(f"\nMaster FE complete. Total features: {X.shape[1]}")
gc.collect()

In [ ]:
# =============================================================================
# BLOCK 3: TWO-STAGE TUNING & TRAINING (MEAN MODEL)
# =============================================================================
print("\n--- STAGE 1, PART 1: Tuning Mean Prediction Model on New Features ---")

# We must re-tune because the feature set has changed significantly.
# A wider search space is appropriate for this new feature set.
def objective_mean(trial):
    train_x, val_x, train_y, val_y = train_test_split(X, y_true, test_size=0.2, random_state=RANDOM_STATE)
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method':'hist',
        'eta': trial.suggest_float('eta', 0.02, 0.06),
        'max_depth': trial.suggest_int('max_depth', 7, 12),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
        'lambda': trial.suggest_float('lambda', 1.0, 8.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7)
    }
    model = xgb.XGBRegressor(**params, n_estimators=2500, random_state=RANDOM_STATE, n_jobs=-1, early_stopping_rounds=100)
    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)
    return np.sqrt(mean_squared_error(val_y, model.predict(val_x)))

# Running a comprehensive search
N_OPTUNA_TRIALS = 50 
study_mean = optuna.create_study(direction='minimize')
study_mean.optimize(objective_mean, n_trials=N_OPTUNA_TRIALS)

best_params_mean = study_mean.best_params
print(f"\n# Mean Model Tuning Complete. Best Validation RMSE: ${study_mean.best_value:,.2f}")

# --- STAGE 1, PART 2: K-Fold Training of Mean Model ---
print("\n# STAGE 1, PART 2: K-Fold Training of Mean Model...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_mean_preds = np.zeros(len(X))
test_mean_preds = np.zeros(len(X_test))
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']
final_params_mean = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 
    'random_state': RANDOM_STATE, 'n_jobs': -1, 
    **best_params_mean
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Mean Model - Fold {fold+1}/{N_SPLITS}...")
    model = xgb.XGBRegressor(**final_params_mean, n_estimators=2500, early_stopping_rounds=100)
    model.fit(X.iloc[train_idx], y_true.iloc[train_idx], eval_set=[(X.iloc[val_idx], y_true.iloc[val_idx])], verbose=False)
    oof_mean_preds[val_idx] = model.predict(X.iloc[val_idx])
    test_mean_preds += model.predict(X_test) / N_SPLITS

final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Mean model K-Fold training complete. Final OOF RMSE: ${final_mean_rmse:,.2f}")
print("-" * 50)

In [ ]:
# =============================================================================
# BLOCK 4: TWO-STAGE TUNING & TRAINING (ERROR MODEL)
# =============================================================================
print("\n--- STAGE 2, PART 1: Tuning Error Prediction Model ---")

# --- Create Feature Set for Error Model ---
error_target = np.abs(y_true - oof_mean_preds)
X_for_error = X.copy()
X_for_error['mean_pred_oof'] = oof_mean_preds
X_test_for_error = X_test.copy()
X_test_for_error['mean_pred_oof'] = test_mean_preds

def objective_error(trial):
    train_x, val_x, train_y, val_y = train_test_split(X_for_error, error_target, test_size=0.2, random_state=RANDOM_STATE)
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist',
        'eta': trial.suggest_float('eta', 0.01, 0.05),
        'max_depth': trial.suggest_int('max_depth', 7, 10),
        'subsample': trial.suggest_float('subsample', 0.8, 0.99),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
        'lambda': trial.suggest_float('lambda', 0.3, 0.8, log=True),
        'alpha': trial.suggest_float('alpha', 0.4, 0.7),
    }
    model = xgb.XGBRegressor(**params, n_estimators=2500, random_state=RANDOM_STATE, n_jobs=-1, early_stopping_rounds=100)
    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)
    return np.sqrt(mean_squared_error(val_y, model.predict(val_x)))

study_error = optuna.create_study(direction='minimize')
study_error.optimize(objective_error, n_trials=N_OPTUNA_TRIALS)
best_params_error = study_error.best_params
print(f"\n# Error Model Tuning Complete. Best Validation RMSE: ${study_error.best_value:,.2f}")

# --- STAGE 2, PART 2: K-Fold Training of Error Model ---
print("\n# STAGE 2, PART 2: K-Fold Training of Error Model...")
oof_error_preds = np.zeros(len(X))
test_error_preds = np.zeros(len(X_test))
final_params_error = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1, **best_params_error}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error, grade_for_stratify)):
    print(f"  Error Model - Fold {fold+1}/{N_SPLITS}...")
    model = xgb.XGBRegressor(**final_params_error, n_estimators=2500, early_stopping_rounds=100)
    model.fit(X_for_error.iloc[train_idx], error_target.iloc[train_idx], eval_set=[(X_for_error.iloc[val_idx], error_target.iloc[val_idx])], verbose=False)
    oof_error_preds[val_idx] = model.predict(X_for_error.iloc[val_idx])
    test_error_preds += model.predict(X_test_for_error) / N_SPLITS

final_error_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\n# Error model K-Fold training complete. Final OOF RMSE: ${final_error_rmse:,.2f}")
print("-" * 50)

In [ ]:
# =============================================================================
# BLOCK 5: FINAL ASYMMETRIC CALIBRATION AND SUBMISSION
# =============================================================================
print("\n--- Final Asymmetric Calibration ---")

oof_error_final = np.clip(oof_error_preds, 0, None) 
best_a, best_b, best_metric = 2.0, 2.0, float('inf')

for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_mean_preds - oof_error_final * a
        high = oof_mean_preds + oof_error_final * b
        metric, coverage = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA, return_coverage=True)
        if metric < best_metric:
            best_metric = metric
            best_a, best_b = a, b

print(f"\nGrid search complete. Final OOF Score: {best_metric:,.2f}. Best multipliers: a={best_a:.2f}, b={best_b:.2f}")

# --- Create Final Submission ---
print("\nCreating final submission file...")
test_error_final = np.clip(test_error_preds, 0, None)
final_lower = test_mean_preds - test_error_final * best_a
final_upper = test_mean_preds + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper)

submission_df = pd.DataFrame({'id': test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
submission_df.to_csv('submission_v2_advanced.csv', index=False)
print("\n'submission_v2_advanced.csv' created successfully!")
display(submission_df.head())